# CPE15 - Week 4: pandas Data Organization and Wrangling

**Programming for Data Science | Professional Elective 1 | AY 2026-2027**

| Syllabus element | Alignment |
|---|---|
| Course outcome | CO2 |
| Course objective | COBJ2 |
| Assessment connection | Quiz 3 and pandas data manipulation activity |
| Sustainable Development Goals | 5, 9, 10 |

This notebook is designed for explanation, live coding, guided practice, and independent follow-through. Run it from top to bottom in a fresh kernel so that every result can be reproduced.


## Lecture map

1. Labeled data structures
2. Cleaning types and text
3. Selection and filtering
4. Derived columns, drop, set index, and reset index
5. Missing data
6. GroupBy: split, apply, combine
7. Merge and concatenate

    **Live-teaching rhythm:** define the idea → predict the result → run a focused example → inspect the saved output → explain the evidence → complete the practice task.

## Learning outcomes

   By the end of the session, students should be able to:

- construct and inspect Series and DataFrames with labeled axes;
- select, filter, add, drop, set, and reset data using explicit rules;
- detect and handle missing or invalid values without hiding data loss;
- aggregate with GroupBy and integrate tables using merge and concatenate;
- produce a reproducible data-quality and summary report.

## Lecture route

1. Series, DataFrames, and inspection
2. Selection, filtering, columns, and indexes
3. Missing data and validation
4. GroupBy, merge, and concatenate
5. An energy-monitoring case study


In [ ]:
from pathlib import Path
import platform
import random
import sys

random.seed(15)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.system()}")
print(f"Artifacts folder: {ARTIFACTS.resolve()}")


> **Reproducibility habit:** a notebook is not finished merely because it ran once. It should run in order from a restarted kernel, use explicit inputs, avoid hidden state, and explain the meaning of its outputs.


## 1. Labeled data structures

A pandas `Series` is a one-dimensional labeled array. A `DataFrame` is a two-dimensional table whose rows and columns have labels. Labels make operations easier to read, but they also introduce alignment behavior: pandas often aligns by label rather than by position.

Begin every unfamiliar dataset by inspecting shape, column names, types, missingness, and a few records.


In [ ]:
import numpy as np
import pandas as pd

energy = pd.DataFrame(
    {
        "timestamp": [
            "2026-08-03 08:00", "2026-08-03 09:00", "2026-08-03 10:00",
            "2026-08-04 08:00", "2026-08-04 09:00", "2026-08-04 10:00",
        ],
        "building": ["CEN", "CEN", "Library", "CEN", "Library", "Library"],
        "voltage_v": [229.1, 231.0, 228.4, 230.2, 999.0, 229.5],
        "current_a": [12.4, 13.1, 8.7, None, 9.3, 9.0],
        "status": ["ok", "OK ", "ok", "missing", "check", "ok"],
    }
)

energy


### Structural inspection before cleaning

The displayed DataFrame lets us see sample records; this continuation makes the structural audit explicit. `shape` reports row and column count, `columns` identifies the available fields, `dtypes` reveals how pandas currently represents them, and `isna().sum()` counts recognized missing values. These checks determine which cleaning operations are needed next.

In [ ]:
print("Shape:", energy.shape)
print("Columns:", energy.columns.tolist())
print("Dtypes:", energy.dtypes, sep=chr(10))
print("Missing values:", energy.isna().sum(), sep=chr(10))


### Discussion clinic — Series and DataFrame labels carry meaning

A `Series` is a one-dimensional labeled array; a `DataFrame` is a two-dimensional table whose columns may have different dtypes. The index identifies rows but is not automatically a unique business key. Before analysis, define what one row represents, whether rows are unique at that granularity, which columns are identifiers or measurements, and the units of each measurement. `head()`, `shape`, `dtypes`, and missing-value counts answer different structural questions.

**Interpretation standard.** The displayed table is not yet analysis-ready: the timestamp is text, status labels are inconsistent, one current is missing, and one voltage is implausible. Inspection should identify these issues before summaries are computed.

### 1.1 Individual topic — `Series`

**What it is and how it works.** A Series is a one-dimensional labeled array with an index and one dtype. It can represent one measured variable or one labeled vector.

**Core syntax**

```python
`pd.Series(values, index=labels, name='variable')`
```

**When to use it.** Use it for one labeled variable, aligned arithmetic, or a single table column.

**When to use another approach.** Use a DataFrame when several variables with distinct names or dtypes belong together.

In [ ]:
# Demonstration — `Series`
topic_voltage_series = pd.Series([229.1, 231.0, 228.4], index=["CEN", "Lab", "Library"], name="voltage_v")
print(topic_voltage_series)
print("mean voltage:", round(topic_voltage_series.mean(), 2))

**Expected output pattern and interpretation.** The index labels identify buildings, the Series name records the unit-bearing variable, and the mean is calculated from three values.

**Science-communication statement.** Say 'mean of three building voltage values' and show the unit; do not imply the buildings form a representative population.

### 1.2 Individual topic — `DataFrame`

**What it is and how it works.** A DataFrame is a two-dimensional labeled table whose columns may have different dtypes. Each row should have a defined granularity.

**Core syntax**

```python
`pd.DataFrame({'column': values, ...})`
```

**When to use it.** Use it for records with several named variables that must be filtered, joined, grouped, or summarized.

**When to use another approach.** Do not treat every rectangular object as analysis-ready; row meaning, keys, types, units, and missingness must be defined.

In [ ]:
# Demonstration — `DataFrame`
topic_frame = pd.DataFrame({"station": ["A", "B"], "temperature_c": [28.4, 29.1], "active": [True, False]})
print(topic_frame)
print(topic_frame.dtypes)

**Expected output pattern and interpretation.** Two station records form the rows, while text, floating, and Boolean columns retain different dtypes.

**Science-communication statement.** Define one row as one station observation and state the time scope; a table display alone does not establish provenance.

### 1.3 Individual topic — Structural inspection

**What it is and how it works.** Inspection methods reveal different aspects of a table: `head` previews rows, `shape` reports dimensions, `dtypes` reports storage types, and `isna` identifies recognized missingness.

**Core syntax**

```python
`df.head()`; `df.shape`; `df.dtypes`; `df.isna().sum()`
```

**When to use it.** Use them immediately after loading or constructing a table and after major transformations.

**When to use another approach.** Do not infer full-data quality from a five-row preview.

In [ ]:
# Demonstration — Structural inspection
topic_inspection = {
    "shape": energy.shape,
    "columns": energy.columns.tolist(),
    "missing": energy.isna().sum().to_dict(),
    "duplicate_rows": int(energy.duplicated().sum()),
}
topic_inspection

**Expected output pattern and interpretation.** The dictionary records table dimensions, fields, missing counts, and exact duplicate count for the full teaching table.

**Science-communication statement.** Report these as structural diagnostics, then explain which findings require cleaning; do not call the dataset clean merely because duplicates are zero.

### Science communication lens

- **Audience:** A data user who needs to know what one row represents and how cleaning changed the available evidence.
- **Lead with the meaning:** The displayed table is not yet analysis-ready: the timestamp is text, status labels are inconsistent, one current is missing, and one voltage is implausible.
- **Show the evidence:** Report source and result row counts, column definitions and units, missingness, conversion failures, filters, join cardinality, and aggregation denominators.
- **State the boundary:** Cleaning and imputation improve usability but can change the represented population; disclose exclusions and avoid describing imputed values as observations.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Create a labeled Series and a mixed-type DataFrame for three campus stations, then inspect labels, shape, dtypes, missingness, and duplicate rows before cleaning.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Labeled data structures
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

import pandas as pd

s = pd.Series(
    [False, False, True],
    index=["CEN","CAM","CABHA"],
    name="has_elevator"
)

print(s)

df = pd.DataFrame({
    "station_name":["CEN", "CAM", "CABHA"],
    "floors":      [3,      None,  None],
    "has_new_cr":  [True, False, False]
})

print(f"\nColumuns:\t{df.columns.tolist()}")
print(f"\nShape:\t{df.shape}")
print(f"\ntype:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isna().sum()}")
print(f"\nDuplicate:\n{df.duplicated().sum()}")



## 2. Cleaning types and text

Converting a timestamp column enables date-aware filtering and grouping. Text fields should be normalized only with a documented purpose. For example, trimming whitespace and standardizing case may be appropriate for category labels, while altering free-text responses may destroy meaning.


In [ ]:
energy["timestamp"] = pd.to_datetime(energy["timestamp"])
energy["status"] = energy["status"].str.strip().str.lower()
energy["date"] = energy["timestamp"].dt.date
energy.dtypes


### Additional worked case — clean text, time, and numeric types with explicit failure behavior

String cleaning standardizes labels; datetime conversion enables time-aware operations; numeric conversion determines whether arithmetic is possible. `errors='coerce'` converts invalid numeric text to `NaN`, which preserves row count but creates missing evidence that must be audited. `errors='raise'` stops immediately. The appropriate choice depends on whether the workflow is exploratory, graded, or production-critical.

In [ ]:
clinic_raw = pd.Series([" 3.91 ", "not_available", "3.44"], name="battery_v")
clinic_numeric = pd.to_numeric(clinic_raw.str.strip(), errors="coerce")

print(pd.DataFrame({"raw": clinic_raw, "numeric": clinic_numeric}))
print("Conversion failures:", int(clinic_numeric.isna().sum()))

**Result and interpretation.** Two values become floats and the invalid text becomes `NaN`. The missing count records one conversion failure; silently dropping that row would hide a data-quality decision.

### 2.1 Individual topic — Datetime conversion

**What it is and how it works.** `pd.to_datetime` parses date/time text into a datetime dtype that supports sorting, extraction, differences, and resampling.

**Core syntax**

```python
`pd.to_datetime(series, errors='raise')`
```

**When to use it.** Use it when text represents timestamps and time-aware operations are required.

**When to use another approach.** Do not assume ambiguous day-month order or time zone; pass format and zone information when known.

In [ ]:
# Demonstration — Datetime conversion
topic_time_text = pd.Series(["2026-08-03 08:00", "2026-08-03 09:30"])
topic_times = pd.to_datetime(topic_time_text, format="%Y-%m-%d %H:%M")
print(topic_times)
print("elapsed minutes:", (topic_times.iloc[1] - topic_times.iloc[0]).total_seconds() / 60)

**Expected output pattern and interpretation.** The text becomes datetime values and the elapsed interval is 90 minutes.

**Science-communication statement.** State the timestamp format and assumed time zone; conversion does not correct an inaccurately recorded time.

### 2.2 Individual topic — String cleaning with `.str` methods

**What it is and how it works.** The `.str` accessor applies vectorized string methods to a Series while preserving index alignment.

**Core syntax**

```python
`series.str.strip().str.lower()`
```

**When to use it.** Use it to standardize case, whitespace, patterns, or substrings before grouping and matching.

**When to use another approach.** Avoid aggressive normalization that merges genuinely different categories; preserve raw labels for audit.

In [ ]:
# Demonstration — String cleaning with `.str` methods
topic_raw_status = pd.Series([" OK ", "ok", "Check", None], name="status")
topic_clean_status = topic_raw_status.str.strip().str.lower()
print(pd.DataFrame({"raw": topic_raw_status, "clean": topic_clean_status}))

**Expected output pattern and interpretation.** The two OK variants standardize to `ok`, `Check` becomes `check`, and missingness remains missing.

**Science-communication statement.** Explain the exact normalization and report how many labels changed; standardization improves consistency but does not validate category truth.

### 2.3 Individual topic — Numeric conversion

**What it is and how it works.** `pd.to_numeric` converts numeric-looking text. `errors='coerce'` turns failures into `NaN`; `errors='raise'` stops at the first failure.

**Core syntax**

```python
`pd.to_numeric(series, errors='coerce')`
```

**When to use it.** Use coercion when conversion failures should be retained and audited as missing values.

**When to use another approach.** Avoid silently dropping failures or replacing them with zero without justification.

In [ ]:
# Demonstration — Numeric conversion
topic_raw_numeric = pd.Series(["3.91", "not_available", "3.44"])
topic_numeric = pd.to_numeric(topic_raw_numeric, errors="coerce")
print(pd.DataFrame({"raw": topic_raw_numeric, "numeric": topic_numeric}))
print("failures:", int(topic_numeric.isna().sum()))

**Expected output pattern and interpretation.** Two values convert to floats and one becomes `NaN`, producing one auditable conversion failure.

**Science-communication statement.** Report received, successfully converted, and failed counts; coercion describes handling, not a reason the source was invalid.

### Science communication lens

- **Audience:** A data user who needs to know what one row represents and how cleaning changed the available evidence.
- **Lead with the meaning:** Two values become floats and the invalid text becomes `NaN`.
- **Show the evidence:** Report source and result row counts, column definitions and units, missingness, conversion failures, filters, join cardinality, and aggregation denominators.
- **State the boundary:** Cleaning and imputation improve usability but can change the represented population; disclose exclusions and avoid describing imputed values as observations.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Clean a column containing padded numeric text and one invalid token using string methods and `pd.to_numeric(errors='coerce')`; count and preserve conversion failures.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Cleaning types and text
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

import pandas as pd

raw_column = pd.Series(
    [" 23  ", " not available ", "10000     "],
    index=["Ben 23", "Ben 10", "Ben 10K"],
    name="number_of_aliens"
)

clean_column = raw_column.str.strip();
clean_column = pd.to_numeric(clean_column, errors="coerce")

print(f"raw column:\n{raw_column}")
print(f"\ncleaned column:\n{clean_column}")
print(f"\n missing:\t{clean_column.isna().sum()}")

## 3. Selection and filtering

Use `.loc[row_labels, column_labels]` for label-based selection and `.iloc[row_positions, column_positions]` for position-based selection. Boolean filtering should state the real rule. Keep a count of excluded rows so cleaning remains auditable.


In [ ]:
selected = energy.loc[:, ["timestamp", "building", "voltage_v", "current_a"]]
cen_rows = energy.loc[energy["building"].eq("CEN")]
first_three = energy.iloc[:3, :4]

print("CEN rows:", len(cen_rows))
print(selected)
print(cen_rows)
first_three


### Convert selection rules into a reusable quality mask

The first selection example demonstrated `.loc` and `.iloc`. This continuation creates two domain rules and combines them into `quality_ok`. Missing current is temporarily allowed so it can be handled explicitly in the missing-data section; the implausible 999 V row is rejected. The new Boolean column preserves the decision for inspection.

In [ ]:
valid_voltage = energy["voltage_v"].between(180, 260)
valid_current = energy["current_a"].between(0, 100) | energy["current_a"].isna()
energy["quality_ok"] = valid_voltage & valid_current

energy.loc[:, ["building", "voltage_v", "current_a", "quality_ok"]]


### Discussion clinic — selection syntax communicates whether labels or positions drive the choice

Use `.loc` for label-based row and column selection and `.iloc` for integer positions. A Boolean mask states a data rule, such as `building == 'CEN'`; a positional slice states where rows happen to appear. Chained indexing can create ambiguous copies and assignments, so prefer one `.loc[row_rule, columns]` operation. After filtering, reconcile the result count and inspect boundary records.

**Interpretation standard.** Selection should be described as a rule, not merely as syntax. 'Retain CEN observations' is reproducible; 'take these three rows' may depend on accidental ordering.

### 3.1 Individual topic — `.loc` label-based selection

**What it is and how it works.** `.loc` selects rows and columns by labels or Boolean conditions. Label slices are inclusive when labels are ordered and present.

**Core syntax**

```python
`df.loc[row_condition, ['column_a', 'column_b']]`
```

**When to use it.** Use it when selection rules refer to names, categories, or Boolean masks.

**When to use another approach.** Avoid chained indexing for assignment; combine the row and column choice in one `.loc` expression.

In [ ]:
# Demonstration — `.loc` label-based selection
topic_cen_voltage = energy.loc[energy["building"].eq("CEN"), ["timestamp", "voltage_v"]]
print(topic_cen_voltage)

**Expected output pattern and interpretation.** Only CEN rows and the two requested columns remain. The selection expresses a labeled rule rather than a position.

**Science-communication statement.** Say how many CEN rows were retained and which fields were shown; do not imply other buildings were invalid.

### 3.2 Individual topic — `.iloc` position-based selection

**What it is and how it works.** `.iloc` selects by zero-based integer positions, and slice stop positions are excluded like ordinary Python slicing.

**Core syntax**

```python
`df.iloc[row_positions, column_positions]`
```

**When to use it.** Use it for previews, algorithmic partitions, or data whose physical position is intentionally meaningful.

**When to use another approach.** Avoid it for business rules that should survive sorting or inserted columns.

In [ ]:
# Demonstration — `.iloc` position-based selection
topic_first_block = energy.iloc[:2, :3]
print(topic_first_block)
print("shape:", topic_first_block.shape)

**Expected output pattern and interpretation.** The first two rows and first three columns produce a `(2, 3)` preview.

**Science-communication statement.** Call this a positional preview, not a representative sample, unless the ordering and sampling design justify representativeness.

### 3.3 Individual topic — Boolean filtering in pandas

**What it is and how it works.** A Boolean Series aligned to the DataFrame index selects rows where the rule is true. Methods such as `.between`, `.eq`, `.isin`, and `.isna` make rules readable.

**Core syntax**

```python
`mask = df['value'].between(low, high)`; `df.loc[mask]`
```

**When to use it.** Use it for explicit domain, category, missingness, or time-window rules.

**When to use another approach.** Avoid losing the mask definition and rejected records after filtering.

In [ ]:
# Demonstration — Boolean filtering in pandas
topic_voltage_mask = energy["voltage_v"].between(180, 260)
topic_voltage_result = energy.loc[topic_voltage_mask, ["building", "voltage_v"]]
print(topic_voltage_result)
print("retained/rejected:", int(topic_voltage_mask.sum()), int((~topic_voltage_mask).sum()))

**Expected output pattern and interpretation.** Five rows fall in the illustrative interval and the 999 V row is rejected.

**Science-communication statement.** State that one of six records fell outside the chosen plausible range; do not automatically label it sensor failure without investigation.

### Science communication lens

- **Audience:** A data user who needs to know what one row represents and how cleaning changed the available evidence.
- **Lead with the meaning:** Selection should be described as a rule, not merely as syntax.
- **Show the evidence:** Report source and result row counts, column definitions and units, missingness, conversion failures, filters, join cardinality, and aggregation denominators.
- **State the boundary:** Cleaning and imputation improve usability but can change the represented population; disclose exclusions and avoid describing imputed values as observations.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Select the same DataFrame subset once with `.loc`, once with `.iloc`, and once with a Boolean mask; explain whether labels, positions, or conditions define each result.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Selection and filtering
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.
import pandas as pd
import numpy as np

energy = pd.DataFrame(
    {
        "timestamp": [
            "2026-08-03 08:00", "2026-08-03 09:00", "2026-08-03 10:00",
            "2026-08-04 08:00", "2026-08-04 09:00", "2026-08-04 10:00",
        ],
        "building": ["CEN", "CEN", "Library", "CEN", "Library", "Library"],
        "voltage_v": [229.1, 231.0, 228.4, 230.2, 999.0, 229.5],
        "current_a": [12.4, 13.1, 8.7, None, 9.3, 9.0],
        "status": ["ok", "OK ", "ok", "missing", "check", "ok"],
    }
)


# location
energy_loc = energy.loc[:, ["timestamp", "building"]]
print(f"LOCATION:\n{energy_loc}")

# iloc
energy_iloc = energy.iloc[:2, :4]
print(f"\nINDEXED LOCATION:\n{energy_iloc}")

# masked filter
# - current: 0 - 10

current_range = energy["current_a"].between(0,10)
current_exists = ~energy["current_a"].isna()
current_mask = current_range & current_exists
energy_masked = energy.loc[current_mask]
print(f"\nMASKED:\n{energy_masked}")

# Write your solution below.

## 4. Derived columns, drop, set index, and reset index

Derived columns should include units in the name when practical. For an approximate single-phase example, apparent power is voltage multiplied by current. Missing current remains missing rather than being silently treated as zero.


In [ ]:
energy["apparent_power_va"] = energy["voltage_v"] * energy["current_a"]
indexed = energy.set_index("timestamp").sort_index()
restored = indexed.reset_index()

print(indexed)
print(restored)

# remove timestamp column
compact = restored.drop(columns=["timestamp"])
compact.head()

### Additional worked case — derived columns, index changes, and dropped fields serve different purposes

A derived column records a transformation while preserving the source fields needed to verify it. `set_index` changes row labels but does not remove information unless `drop=True` applies to the source column; `reset_index` restores the index to an ordinary column. `drop` removes labels and should be delayed until their audit value is no longer needed. Prefer assigning units in names such as `_va` or `_kwh`.

In [ ]:
clinic_energy = energy.assign(
    power_kw=lambda frame: frame["voltage_v"] * frame["current_a"] / 1000,
    hour=lambda frame: frame["timestamp"].dt.hour,
)

print(clinic_energy[["timestamp", "power_kw", "hour"]].head())
print("Original columns unchanged:", "power_kw" not in energy.columns)

**Result and interpretation.** `.assign()` returns a new DataFrame containing the two derived columns, while the original `energy` table remains unchanged. This makes comparison and testing easier during a teaching workflow.

### 4.1 Individual topic — Derived column

**What it is and how it works.** A derived column stores a calculation from existing fields while retaining row alignment. Unit-bearing names make formulas auditable.

**Core syntax**

```python
`df['power_w'] = df['voltage_v'] * df['current_a']`
```

**When to use it.** Use it when the derived quantity will be inspected, reused, grouped, or exported.

**When to use another approach.** Avoid overwriting source measurements or hiding unit conversion inside an ambiguous name.

In [ ]:
# Demonstration — Derived column
topic_derived = energy[["voltage_v", "current_a"]].copy()
topic_derived["apparent_power_va"] = topic_derived["voltage_v"] * topic_derived["current_a"]
print(topic_derived.head())

**Expected output pattern and interpretation.** Each nonmissing row receives apparent power as voltage times current; a missing current produces a missing derived value.

**Science-communication statement.** Name the formula, units, and missing-value propagation; do not describe apparent power as real power without the required phase information.

### 4.2 Individual topic — `drop`

**What it is and how it works.** `drop` removes labeled rows or columns and returns a new object unless `inplace=True` is used.

**Core syntax**

```python
`df.drop(columns=['temporary_column'])`
```

**When to use it.** Use it to create a deliberate interface after fields are no longer needed for verification.

**When to use another approach.** Avoid dropping provenance, raw values, or rejection reasons before audit is complete.

In [ ]:
# Demonstration — `drop`
topic_with_note = energy.assign(temporary_note="teaching")
topic_without_note = topic_with_note.drop(columns=["temporary_note"])
print("before:", topic_with_note.columns.tolist())
print("after:", topic_without_note.columns.tolist())

**Expected output pattern and interpretation.** Only the temporary column is removed, while the original `energy` table remains unchanged.

**Science-communication statement.** List the removed fields and reason; absence from an output should not obscure that the field existed in source data.

### 4.3 Individual topic — `set_index` and `reset_index`

**What it is and how it works.** `set_index` moves one or more columns into row labels; `reset_index` returns those labels to columns and creates a default integer index.

**Core syntax**

```python
`df.set_index('timestamp')`; `indexed.reset_index()`
```

**When to use it.** Use an index for alignment, time-oriented selection, or display when it matches row identity.

**When to use another approach.** Avoid assuming an index is unique; verify uniqueness when it serves as a key.

In [ ]:
# Demonstration — `set_index` and `reset_index`
topic_indexed = energy.set_index("timestamp")
topic_restored = topic_indexed.reset_index()
print("index name:", topic_indexed.index.name)
print("index unique:", topic_indexed.index.is_unique)
print("restored first column:", topic_restored.columns[0])

**Expected output pattern and interpretation.** Timestamp becomes the index and is not unique in this table; reset restores it as the first column.

**Science-communication statement.** Report that timestamps repeat across buildings, so timestamp alone is not a unique record key.

### Science communication lens

- **Audience:** A data user who needs to know what one row represents and how cleaning changed the available evidence.
- **Lead with the meaning:** `.assign()` returns a new DataFrame containing the two derived columns, while the original `energy` table remains unchanged.
- **Show the evidence:** Report source and result row counts, column definitions and units, missingness, conversion failures, filters, join cardinality, and aggregation denominators.
- **State the boundary:** Cleaning and imputation improve usability but can change the represented population; disclose exclusions and avoid describing imputed values as observations.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Create a unit-bearing derived column, remove one temporary field with `drop`, set a timestamp index, and reset it while checking that the row count stays unchanged.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Derived columns, drop, set index, and reset index
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

import pandas as pd
import numpy as np

energy = pd.DataFrame(
    {
        "timestamp": [
            "2026-08-03 08:00", "2026-08-03 09:00", "2026-08-03 10:00",
            "2026-08-04 08:00", "2026-08-04 09:00", "2026-08-04 10:00",
        ],
        "building": ["CEN", "CEN", "Library", "CEN", "Library", "Library"],
        "voltage_v": [229.1, 231.0, 228.4, 230.2, 999.0, 229.5],
        "current_a": [12.4, 13.1, 8.7, None, 9.3, 9.0],
        "status": ["ok", "OK ", "ok", "missing", "check", "ok"],
    }
)

# derive the power from voltage and current
power = energy["current_a"] * energy["voltage_v"]
energy["power"] = power

# drop the current and voltage column
energy_no_volt_and_current = energy.drop(columns=["voltage_v", "current_a"])
energy_no_volt_and_current

# set timestamp index
timestamp_indexed = energy.set_index("timestamp").sort_index()
restored = timestamp_indexed.reset_index()

print(f"derived:\n{energy}")
print(f"\n\ndropped:\n{energy_no_volt_and_current}")
print(f"\n\nrestored:\n{restored}")


## 5. Missing data

Missing values can mean unavailable, not applicable, not measured, or removed. The correct treatment depends on why a value is missing. Demonstration methods such as dropping or filling should be accompanied by counts and a justification.


In [ ]:
missing_report = energy.isna().sum().rename("missing_count").to_frame()
missing_report["missing_pct"] = (
    missing_report["missing_count"] / len(energy) * 100
).round(1)
missing_report


### Apply the quality rule and treat missing current within groups

The missingness report identifies where evidence is incomplete. This continuation retains rows that passed the earlier quality mask, copies them to avoid ambiguous assignment, and fills missing current with the median from the same building. The printed retained count makes the filtering denominator visible; imputation remains a modeling choice requiring justification.

In [ ]:
clean_energy = energy.loc[energy["status"].str.strip().str.lower().eq("ok")].copy()

clean_energy["current_a"] = clean_energy.groupby("building")["current_a"].transform(
    lambda values: values.fillna(values.median())
)

clean_energy["apparent_power_va"] = (
    clean_energy["voltage_v"] * clean_energy["current_a"]
)

print(energy)
print(f"Retained {len(clean_energy)} of {len(energy)} rows")
clean_energy


Median imputation is shown for practice, not as a universal rule. The transformed value should be flagged in a real report, and the reason for missingness should be investigated.


### Discussion clinic — missingness is a question about meaning, not only a method call

`isna()` detects missing markers recognized by pandas, but it will not detect undocumented sentinels such as `-999` or the text `'unknown'` unless they are standardized first. Dropping, imputing, or retaining missing values changes the population represented by a calculation. Report missing counts by field and, when relevant, by group or time period. Never replace missing measurements with zero unless zero is a justified observation.

**Interpretation standard.** A missingness report identifies where evidence is incomplete. The next decision must be justified using the field's meaning, the analysis goal, and the potential bias introduced by exclusion or imputation.

### 5.1 Individual topic — Detecting missing values

**What it is and how it works.** `isna` recognizes pandas missing markers and returns a Boolean object; summing Boolean values counts missing entries.

**Core syntax**

```python
`df.isna().sum()`; `df['column'].isna()`
```

**When to use it.** Use it by field and, when relevant, by group or time before choosing treatment.

**When to use another approach.** Avoid assuming undocumented sentinels such as `-999` are missing until standardized.

In [ ]:
# Demonstration — Detecting missing values
topic_missing_by_building = (energy
                             .groupby("building")["current_a"]
                             .apply(
                                 lambda values: int(values.isna().sum())
                            )
)
print(topic_missing_by_building)

**Expected output pattern and interpretation.** The missing-current count is separated by building, showing where the incomplete evidence occurs.

**Science-communication statement.** Report counts and denominators by group; a global percentage can hide concentrated missingness.

### 5.2 Individual topic — Dropping missing rows

**What it is and how it works.** `dropna` removes rows or columns according to missingness rules such as `subset` and `how`.

**Core syntax**

```python
`df.dropna(subset=['required_column'])`
```

**When to use it.** Use it when the required variable is essential and exclusion bias is assessed and acceptable.

**When to use another approach.** Avoid complete-case deletion by habit, especially when missingness is systematic.

In [ ]:
# Demonstration — Dropping missing rows
topic_complete_current = energy.dropna(subset=["current_a"])
print("received:", len(energy))
print("retained:", len(topic_complete_current))
print("excluded:", len(energy) - len(topic_complete_current))

**Expected output pattern and interpretation.** One of six rows is excluded because current is missing, leaving five complete-current rows.

**Science-communication statement.** State the field and exclusion count; do not silently present the remaining rows as the full dataset.

### 5.3 Individual topic — Imputation

**What it is and how it works.** Imputation replaces missing values using a stated rule, such as a group median. The result is estimated rather than observed.

**Core syntax**

```python
`series.fillna(series.median())`; groupwise `transform`
```

**When to use it.** Use it when the analysis requires complete values and assumptions are justified and sensitivity is considered.

**When to use another approach.** Avoid treating imputed values as measured or using future information in time-sensitive prediction.

In [ ]:
# Demonstration — Imputation
topic_imputed = energy[["building", "current_a"]].copy()
topic_imputed["was_imputed"] = topic_imputed["current_a"].isna()
topic_imputed["current_a"] = topic_imputed.groupby("building")["current_a"].transform(lambda values: values.fillna(values.median()))
print(energy.loc[:, ["building","current_a"]])
print(topic_imputed)

**Expected output pattern and interpretation.** The missing CEN current is replaced by the median of observed CEN currents and is flagged by `was_imputed`.

**Science-communication statement.** Disclose method, group, and number imputed; preserve a flag so estimates are not confused with observations.

### Science communication lens

- **Audience:** A data user who needs to know what one row represents and how cleaning changed the available evidence.
- **Lead with the meaning:** A missingness report identifies where evidence is incomplete.
- **Show the evidence:** Report source and result row counts, column definitions and units, missingness, conversion failures, filters, join cardinality, and aggregation denominators.
- **State the boundary:** Cleaning and imputation improve usability but can change the represented population; disclose exclusions and avoid describing imputed values as observations.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Compare dropping incomplete rows with group-median imputation on the same table; flag imputed values and explain how each choice changes the evidence base.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Missing data
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

import pandas as pd
import numpy as np


energy = pd.DataFrame(
    {
        "timestamp": ["2025-08-03 08:00", "2026-08-03 09:00", "2026-08-03 10:00", "2025-08-04 08:00", "2026-08-04 09:00", "2026-08-04 10:00",],
        "building":  ["CEN", "CEN", "Library", "CEN", "Library", "Library"],
        "voltage_v": [228.1, 231.0, 228.4, 230.2, 999.0, 229.5],
        "current_a": [11.4, 13.1, 8.7, None, 9.3, 9.0],
        "status":    ["ok", "OK ", "ok", "missing", "check", "ok"],
    }
)

# method 1: drop
energy_dropped = energy.dropna(subset=["current_a"])
print("Method 1: drop")
print(f"RETAINED: {len(energy_dropped)} out of {len(energy)}")
print(energy_dropped,"\n\n")

# method 2: imputed
energy_imputed = energy.loc[:, ["building", "current_a"]].copy()
energy_imputed["is_imputed"] = energy_imputed["current_a"].isna();
energy_imputed["current_a"] = energy["current_a"].transform(lambda value: value.fillna(value.median()))
print("\nMethod 2: imputation")
print(energy_imputed,"\n\n")
# Write your solution below.

## 6. GroupBy: split, apply, combine

GroupBy partitions rows by one or more keys, applies an aggregation or transformation, and combines the results. Name the output columns so the summary remains readable.


In [ ]:
building_summary = (
    clean_energy.groupby("building", as_index=False)
    .agg(
        observations=("building", "size"),
        mean_voltage_v=("voltage_v", "mean"),
        max_current_a=("current_a", "max"),
        mean_apparent_power_va=("apparent_power_va", "mean"),
    )
    .round(2)
)
building_summary


### Additional worked case — aggregation reduces rows while transform preserves alignment

`groupby(...).agg(...)` normally returns one row per group, changing granularity. `groupby(...).transform(...)` returns a result aligned to each original row, which is useful for deviations from a group mean. Name every aggregation and state its denominator. Grouping by the wrong key can produce plausible but misleading summaries. Compare the output row count with the number of expected groups.

In [ ]:
clinic_grouped = clean_energy.copy()
clinic_grouped["building_mean_voltage_v"] = (
    clinic_grouped.groupby("building")["voltage_v"].transform("mean")
)
clinic_grouped["voltage_deviation_v"] = (
    clinic_grouped["voltage_v"] - clinic_grouped["building_mean_voltage_v"]
)

print(clinic_grouped[["building", "voltage_v", "building_mean_voltage_v", "voltage_deviation_v"]])

**Result and interpretation.** The transformed mean repeats within each building so every observation can be compared with its own group. Unlike the summary table, this result retains the original row count.

### 6.1 Individual topic — `groupby`

**What it is and how it works.** `groupby` partitions rows by one or more keys and creates a grouped object awaiting aggregation, transformation, filtering, or iteration.

**Core syntax**

```python
`grouped = df.groupby('building')`
```

**When to use it.** Use it when a question asks for within-category or within-time summaries.

**When to use another approach.** Avoid grouping by a field that does not represent the intended analytical unit.

In [ ]:
# Demonstration — `groupby`
topic_groups = clean_energy.groupby("building")
print("group names:", list(topic_groups.groups))
print("group sizes:", topic_groups.size().to_dict())

**Expected output pattern and interpretation.** Rows are split into CEN and Library groups, and their observation counts are displayed.

**Science-communication statement.** State the grouping key and number of observations in every group before comparing summaries.

### 6.2 Individual topic — `.agg` aggregation

**What it is and how it works.** `.agg` reduces each group to named summary values, typically changing granularity to one row per group.

**Core syntax**

```python
`groupby(...).agg(mean_value=('value', 'mean'))`
```

**When to use it.** Use it for group-level tables with explicit measures and names.

**When to use another approach.** Avoid unnamed or mixed aggregations whose units and denominators are unclear.

In [ ]:
# Demonstration — `.agg` aggregation
topic_aggregate = clean_energy.groupby("building", as_index=False).agg(
    n=("building", "size"),
    median_voltage_v=("voltage_v", "median"),
)
print(topic_aggregate)

**Expected output pattern and interpretation.** One row per building reports count and median voltage, making the changed granularity explicit.

**Science-communication statement.** Lead with the group comparison and include `n`; avoid implying a median difference is statistically or practically important without context.

### 6.3 Individual topic — `.transform` group-aligned result

**What it is and how it works.** `.transform` computes within groups but returns values aligned to the original rows, preserving row count.

**Core syntax**

```python
`df.groupby(key)[value].transform('mean')`
```

**When to use it.** Use it for group-centered values, imputation, normalization, or group metrics attached to each record.

**When to use another approach.** Use `.agg` instead when one result row per group is the desired output.

In [ ]:
# Demonstration — `.transform` group-aligned result
topic_transform = clean_energy[["building", "voltage_v"]].copy()
topic_transform["group_mean_v"] = topic_transform.groupby("building")["voltage_v"].transform("mean")
topic_transform["deviation_v"] = topic_transform["voltage_v"] - topic_transform["group_mean_v"]
print(topic_transform)

**Expected output pattern and interpretation.** Group means repeat beside their source rows, enabling one deviation per observation without reducing row count.

**Science-communication statement.** Describe deviations relative to each building's observed mean and retain units; do not call them anomalies without a rule.

### Science communication lens

- **Audience:** A data user who needs to know what one row represents and how cleaning changed the available evidence.
- **Lead with the meaning:** The transformed mean repeats within each building so every observation can be compared with its own group.
- **Show the evidence:** Report source and result row counts, column definitions and units, missingness, conversion failures, filters, join cardinality, and aggregation denominators.
- **State the boundary:** Cleaning and imputation improve usability but can change the represented population; disclose exclusions and avoid describing imputed values as observations.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Group energy observations by building and use `agg` for one-row-per-building summaries and `transform` for row-aligned deviations; state the granularity of both outputs.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — GroupBy: split, apply, combine
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

import pandas as pd
import numpy as np

energy = pd.DataFrame(
    {
        "timestamp": ["2025-08-03 08:00", "2026-08-03 09:00", "2026-08-03 10:00", "2025-08-04 08:00", "2026-08-04 09:00", "2026-08-04 10:00",],
        "building":  ["CEN", "CEN", "Library", "CEN", "Library", "Library"],
        "voltage_v": [228.1, 231.0, 228.4, 230.2, 999.0, 229.5],
        "current_a": [11.4, 13.1, 8.7, None, 9.3, 9.0],
        "status":    ["ok", "OK ", "ok", "missing", "check", "ok"],
    }
)

energy_by_building = energy.groupby("building");

# method 1: agg
output_1 = energy_by_building.agg(
    mean_current=("current_a", "mean"),
    mean_voltage=("voltage_v", "mean"),
)

# method 2: transform
output_2 = energy.loc[:, ["building", "voltage_v", "current_a"]].copy()
output_2["current_mean"] = energy_by_building["current_a"].transform("mean")
output_2["voltage_mean"] = energy_by_building["voltage_v"].transform("mean")
output_2 = output_2.drop(columns=["voltage_v", "current_a"])

print("Output 1: (agg)")
print(output_1,)

print("\n\nOutput 2: (transform)")
print(output_2)

## 7. Merge and concatenate

`merge` combines related tables using keys, like a database join. `concat` stacks compatible objects along rows or columns. Before a merge, check whether keys are unique and state the expected relationship: one-to-one, many-to-one, or many-to-many.


In [ ]:
building_info = pd.DataFrame(
    {
        "building": ["CEN", "Library"],
        "floor_area_m2": [2400, 1800],
        "manager": ["Engineering", "Library Services"],
    }
)

enriched = building_summary.merge(
    building_info,
    on="building",
    how="left",
    validate="one_to_one",
)
enriched["va_per_m2"] = (
    enriched["mean_apparent_power_va"] / enriched["floor_area_m2"]
)
enriched.round(3)


### Concatenation as row-wise recombination

The merge example added building attributes by matching keys. Concatenation answers a different question: it stacks compatible row partitions without relational key matching. Splitting and recombining the clean table demonstrates `pd.concat(..., ignore_index=True)`, followed by row-count and column-set assertions that check structural preservation.

In [ ]:
morning = clean_energy.iloc[:2].copy()
later = clean_energy.iloc[2:].copy()
recombined = pd.concat([morning, later], ignore_index=True)

assert len(recombined) == len(clean_energy)
assert set(recombined.columns) == set(clean_energy.columns)
recombined.head()


### Discussion clinic — merge combines columns; concatenate stacks compatible objects

A merge uses keys to relate records. The join type determines which unmatched keys remain, and `validate=` states expected cardinality such as one-to-one or many-to-one. `concat` does not discover relational matches; along rows it stacks records with aligned columns, and along columns it aligns indexes. Always compare source and result row counts, inspect unmatched keys, and check whether duplicates multiplied rows.

**Interpretation standard.** The enriched table should have one row per building because both inputs are one row per building and the merge is validated one-to-one. A correct-looking value does not replace cardinality checks.

### 7.1 Individual topic — `merge`

**What it is and how it works.** `merge` combines columns by matching key values. Join type controls unmatched rows and `validate` checks expected key cardinality.

**Core syntax**

```python
`left.merge(right, on='key', how='left', validate='many_to_one')`
```

**When to use it.** Use it for relationally linking records and attributes through documented keys.

**When to use another approach.** Avoid joining on labels that are nonunique, inconsistent, or semantically different without reconciliation.

In [ ]:
# Demonstration — `merge`
topic_left = pd.DataFrame({"station": ["A", "B", "C"], "value": [10, 20, 30]})
topic_right = pd.DataFrame({"station": ["A", "B"], "region": ["North", "South"]})
topic_merged = topic_left.merge(topic_right, on="station", how="left", validate="many_to_one", indicator=True)
print(topic_merged)

**Expected output pattern and interpretation.** All three left rows remain; station C has no matching region and is marked `left_only`.

**Science-communication statement.** Report matched and unmatched key counts; do not silently treat an unmatched attribute as a measured missing value.

### 7.2 Individual topic — `concat`

**What it is and how it works.** `concat` stacks or aligns pandas objects along an axis without relational key matching. Along rows, columns align by name.

**Core syntax**

```python
`pd.concat([frame_a, frame_b], ignore_index=True)`
```

**When to use it.** Use it for compatible partitions with the same row meaning and schema.

**When to use another approach.** Avoid concatenating different granularities or units merely because column names match.

In [ ]:
# Demonstration — `concat`
topic_batch_a = pd.DataFrame({"station": ["A"], "temperature_c": [28.4]})
topic_batch_b = pd.DataFrame({"station": ["B"], "temperature_c": [29.1]})
topic_stacked = pd.concat([topic_batch_a, topic_batch_b], ignore_index=True)
print(topic_stacked)

**Expected output pattern and interpretation.** Two one-row batches become one two-row table with aligned station and temperature columns.

**Science-communication statement.** State that batches shared schema, units, and granularity; concatenation does not verify duplicate or overlapping observations.

### Science communication lens

- **Audience:** A data user who needs to know what one row represents and how cleaning changed the available evidence.
- **Lead with the meaning:** The enriched table should have one row per building because both inputs are one row per building and the merge is validated one-to-one.
- **Show the evidence:** Report source and result row counts, column definitions and units, missingness, conversion failures, filters, join cardinality, and aggregation denominators.
- **State the boundary:** Cleaning and imputation improve usability but can change the represented population; disclose exclusions and avoid describing imputed values as observations.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Perform a left merge with an indicator column and a row-wise concatenation of compatible batches; reconcile matched, left-only, and total result counts.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Merge and concatenate
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

import pandas as pd

# method 1: merge
station_values = pd.DataFrame({"station": ["A", "B", "C"], "value": [10, 20, 30]})
station_regions = pd.DataFrame({"station": ["A", "B"], "region": ["North", "South"]})

station_merged = station_values.merge(
    station_regions,
    on="station",
    how="left",
    validate="one_to_one"
)

# method 2 concat
record_1 = pd.DataFrame({
    "station": ["A", "B"],
    "value": [10, 20]
})

record_2 = pd.DataFrame({
    "station": ["C"],
    "value": [50]
})

station_concat = pd.concat([record_1, record_2])

print(f"Method 1: merge\n{station_merged}")
print(f"left table rows: {len(left)}   |  right table rows: {len(right)}   |   merged rows count: {len(station_merged)}")
print(f"\nMethod 2: concat\n{station_concat}")
print(f"1st record: {len(record_1)}   |  2nd record: {len(record_2)}   |   merged rows count: {len(station_concat)}")
# Write your solution below.

## Common mistakes

- Chained assignment that may update a temporary view; use `.loc` or `.copy()` explicitly.
- Dropping missing rows without reporting how many were lost.
- Merging on dirty or nonunique keys and accidentally multiplying rows.
- Aggregating before confirming units and granularity.
- Resetting an index and unintentionally creating an unwanted `index` column.


In [ ]:
quality_report = {
    "input_rows": len(energy),
    "retained_rows": len(clean_energy),
    "excluded_rows": len(energy) - len(clean_energy),
    "remaining_missing": int(clean_energy.isna().sum().sum()),
    "duplicate_rows": int(clean_energy.duplicated().sum()),
}
assert quality_report["remaining_missing"] == 0
quality_report


## Independent challenge

Create a DataFrame with at least twelve observations from two or more devices. Include one invalid value, one missing value, and a metadata table. Clean and validate the observations, create two derived columns, produce a GroupBy summary, merge the metadata with validation enabled, and write an interpretation that reports retained and excluded rows.

**Submission expectation:** include readable code, meaningful variable names, a short interpretation, and evidence that the notebook was restarted and run from top to bottom.


In [27]:
import numpy as np
import pandas as pd

# 12 observations, 2 devices: one invalid (99.9), one missing (None)
sensor = pd.DataFrame(
    {
        "device": ["D1"] * 6 + ["D2"] * 6,
        "reading": [10.1, 10.3, 10.2, 99.9, 10.4, None,
                    20.1, 20.2, 20.3, 20.4, 20.1, 20.5],
        "current_a": [1.1, 1.2, 1.3, 1.4, 1.5, 1.6,
                      2.1, 2.2, 2.3, 2.4, 2.5, 2.6],
    }
)

metadata = pd.DataFrame(
    {"device": ["D1", "D2"], "location": ["North", "South"], "zone": ["A", "B"]}
)

# remove rows with nan values and execeeding voltage
sensor_clean = sensor.dropna(subset="reading")
reading_nan_count = sensor["reading"].isna().sum()

mask_max_reading = sensor["reading"].between(0,30)
sensor_clean = sensor.loc[mask_max_reading]

# derive row (power)
sensor_clean["power"] = sensor_clean["current_a"] * sensor_clean["reading"]

# group by device
sensor_grouped = sensor_clean.groupby("device")
sensor_summary = sensor_grouped.agg(
    mean_current=("current_a", "mean"),
    median_power=("power", "median")
)

# merge with metadata table
sensor_summary = sensor_summary.merge(
    metadata,
    on="device",
    how="left",
    validate="one_to_one"
)

print(sensor_summary);

  device  mean_current  median_power location zone
0     D1         1.275        12.810    North    A
1     D2         2.350        47.825    South    B


## Key takeaways

            - pandas combines array computation with meaningful row and column labels.
- Inspection and validation should precede transformation and aggregation.
- GroupBy summarizes within clearly defined groups; merge connects tables through keys.
- Every cleaning decision should preserve counts, rationale, and limitations.

            ## Exit ticket

            1. When should `.loc` be preferred over `.iloc`?
2. Why should a merge specify the expected key relationship?
3. What information must accompany an imputation decision?


## References and further reading

            - McKinney, W. (2022). Python for Data Analysis (3rd ed.).
- pandas development team. pandas User Guide.
- CPE15 syllabus, Week 4 course learning plan.

            <details>
            <summary><strong>Instructor facilitation note</strong></summary>

            Ask students to predict outputs before execution, compare at least two valid approaches, and explain results in plain language. During live coding, deliberately trigger one common error and model a calm debugging process.

            </details>
